# 07 · Validación del detector de agotados

El detector (`src/censura.py`) es la pieza central del proyecto: la ablación de `06` muestra que el ahorro de inventario viene de corregir la demanda censurada. Sin registros de inventario no hay etiquetas reales, así que se valida con dos experimentos:

- **A · Sensibilidad.** Se insertan agotados artificiales (rachas de 3 a 28 días en cero) en periodos limpios de las series reales y se mide qué fracción detecta.
- **B · Falsas alarmas.** Se generan series *sin agotados* con el mismo nivel de venta (media móvil de 28 días) y la misma sobredispersión (binomial negativa) de cada producto, y se cuenta cuántos días marca como agotado.

In [1]:
from pathlib import Path
import sys, time
from datetime import timedelta
import numpy as np
import polars as pl
from censura import clasificar_ceros

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
PROC = ROOT / "data" / "processed"
REP = ROOT / "reports"
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))
df = pl.read_parquet(PROC / "m5_CA_1.parquet").with_columns(pl.col("id").cast(pl.Utf8))
dev_end = df["date"].max() - timedelta(days=28)
dev = df.filter(pl.col("date") <= dev_end).select(["id", "date", "sales"]).sort(["id", "date"])

t0 = time.time()
base = clasificar_ceros(dev)
print(f"clasificar_ceros: {time.time() - t0:.1f}s")
masc = pl.read_parquet(PROC / "mascara_CA_1.parquet").with_columns(pl.col("id").cast(pl.Utf8))
cmp = base.join(masc, on=["id", "date"], suffix="_02")
print("Coincide con la máscara de 02:", (cmp["tipo"] == cmp["tipo_02"]).mean(), cmp.height, base.height)
base.group_by("tipo").len().with_columns(pct=(pl.col("len") / base.height * 100).round(2)).sort("len", descending=True)

clasificar_ceros: 0.7s


Coincide con la máscara de 02: 1.0 4702895 4702895


tipo,len,pct
str,u32,f64
"""normal""",4103296,87.25
"""inactivo""",466480,9.92
"""agotado""",126416,2.69
"""pre_lanzamiento""",6703,0.14


In [2]:
# ---------- Experimento A: sensibilidad (agotados artificiales en series reales) ----------
VENTANA = 56
con_p0 = (base.join(dev, on=["id", "date"])
          .with_columns(p0=(pl.col("sales") == 0).cast(pl.Float64)
                        .rolling_mean(window_size=VENTANA, min_samples=28).shift(1).over("id"),
                        normal=(pl.col("tipo") == "normal").cast(pl.Int32)))
rng = np.random.default_rng(42)
filas = []
for k in [3, 5, 7, 14, 28]:
    for rep in range(3):
        # días finales e tales que [e-k-56+1, e] es todo normal → la racha [e-k+1, e] cae en historia limpia
        cand = (con_p0.with_columns(ok=pl.col("normal").rolling_sum(VENTANA + k).over("id") == VENTANA + k,
                                    p0_ini=pl.col("p0").shift(k - 1).over("id"))
                .filter(pl.col("ok") & pl.col("p0_ini").is_not_null())
                .select(["id", "date", "p0_ini"]))
        elegidos = (cand.with_columns(u=pl.lit(rng.random(cand.height)))
                    .sort("u").group_by("id").first()
                    .select("id", fin=pl.col("date"), p0=pl.col("p0_ini")))
        dias = (elegidos.with_columns(date=pl.date_ranges(pl.col("fin") - timedelta(days=k - 1), "fin"))
                .explode("date").select(["id", "date", "p0"]).with_columns(inyectado=pl.lit(True)))
        mod = (dev.join(dias, on=["id", "date"], how="left")
               .with_columns(sales=pl.when(pl.col("inyectado")).then(0).otherwise(pl.col("sales"))))
        res = (clasificar_ceros(mod.select(["id", "date", "sales"]))
               .join(dias, on=["id", "date"], how="inner"))
        filas.append(res.select("id", "p0", detectado=pl.col("tipo") == "agotado")
                     .with_columns(k=pl.lit(k), rep=pl.lit(rep)))
A_ = pl.concat(filas).with_columns(
    tramo=pl.col("p0").cut([0.1, 0.3, 0.5], labels=["<10%", "10–30%", "30–50%", ">50%"]))
sens = (A_.pivot(on="k", index="tramo", values="detectado", aggregate_function="mean").sort("tramo")
          .rename({str(k): f"{k}_dias" for k in [3, 5, 7, 14, 28]}))
sens.write_csv(REP / "detector_sensibilidad_CA_1.csv")
print("Días inyectados detectados (productos que venden ≥ 50% de los días):",
      f"{A_.filter(pl.col('p0') <= 0.5)['detectado'].mean():.1%}")
sens  # fila: % de días con venta cero en las 8 semanas previas; columna: duración del agotado

Días inyectados detectados (productos que venden ≥ 50% de los días): 85.7%


/var/folders/ph/rltyxjrx2tjbmr5kwfl8jpb40000gn/T/ipykernel_14781/464110075.py:28: DeprecationWarning: `cut` is deprecated; use `bin_intervals` instead
  tramo=pl.col("p0").cut([0.1, 0.3, 0.5], labels=["<10%", "10–30%", "30–50%", ">50%"]))


tramo,3_dias,5_dias,7_dias,14_dias,28_dias
enum,f64,f64,f64,f64,f64
"""<10%""",0.645161,1.0,1.0,0.998966,1.0
"""10–30%""",0.056376,0.469201,0.941176,0.999351,0.998734
"""30–50%""",0.023198,0.071705,0.210685,0.998993,0.995984
""">50%""",0.00151,0.004129,0.006725,0.025602,0.020013


In [3]:
# ---------- Experimento B: falsas alarmas (series sintéticas sin agotados) ----------
s = (con_p0.sort(["id", "date"])
     .with_columns(limpia=pl.when(pl.col("normal") == 1).then(pl.col("sales").cast(pl.Float64)))
     .with_columns(mu=pl.col("limpia").rolling_mean(window_size=28, min_samples=7).shift(1).over("id"))
     .with_columns(mu=pl.col("mu").fill_null(pl.col("limpia").mean().over("id")))
     .with_columns(mu=pl.when(pl.col("tipo").is_in(["inactivo", "pre_lanzamiento"])).then(0.0)
                   .otherwise(pl.col("mu")).fill_null(0.0)))
phi = (s.filter(pl.col("normal") == 1)
       .group_by("id").agg(phi=(((pl.col("sales") - pl.col("mu")) ** 2 - pl.col("mu")).sum()
                               / (pl.col("mu") ** 2).sum()).clip(lower_bound=0)))
s = s.join(phi, on="id", how="left").with_columns(pl.col("phi").fill_null(0.0))
mu, ph = s["mu"].to_numpy(), s["phi"].to_numpy()
tasas = []
for seed in range(3):
    g = np.random.default_rng(seed)
    lam = np.where(ph > 0, g.gamma(1 / np.maximum(ph, 1e-9), mu * ph), mu)
    y = g.poisson(np.where(mu > 0, lam, 0))
    sint = s.select(["id", "date"]).with_columns(sales=pl.Series(y))
    r = clasificar_ceros(sint)
    tasas.append((r["tipo"] == "agotado").mean() * 100)
    print(f"seed {seed}: agotado en sintético = {tasas[-1]:.2f}% de días-producto")
obs = (base["tipo"] == "agotado").mean() * 100
print(f"Observado: {obs:.2f}% | falsas alarmas ≈ {np.mean(tasas):.2f}% → precisión aprox. {1 - np.mean(tasas) / obs:.0%}")
pl.DataFrame({"semilla": range(3), "agotado_sintetico_pct": tasas, "agotado_observado_pct": obs}) \
  .write_csv(REP / "detector_falsas_alarmas_CA_1.csv")

seed 0: agotado en sintético = 0.36% de días-producto


seed 1: agotado en sintético = 0.37% de días-producto


seed 2: agotado en sintético = 0.36% de días-producto
Observado: 2.69% | falsas alarmas ≈ 0.36% → precisión aprox. 86%


## Conclusiones

- **Sensibilidad.** En productos que venden casi todos los días (< 10% de días en cero) detecta prácticamente todo agotado de 5 días o más. En productos con 10–50% de días en cero necesita rachas más largas: 94% de detección a 7 días en el tramo 10–30% y ~100% a partir de 14 días en todos los tramos hasta 50%.
- **Punto ciego por diseño.** En productos que venden menos de la mitad de los días (`P0_MAX = 0.5`) casi no detecta agotados (≤ 3%): una racha de ceros ahí es indistinguible del azar. Por eso la demanda censurada de los productos lentos queda subestimada y **el efecto de la censura medido en 06 es una cota inferior**.
- **Falsas alarmas.** En series sin agotados marca ~0.36% de los días-producto, contra 2.69% observado en los datos reales: aproximadamente **86% de los agotados detectados serían reales**.
- **Limitación:** las series sintéticas no tienen estacionalidad semanal, promociones ni cambios de surtido, así que las falsas alarmas reales podrían ser algo mayores.